# CNN-GNN HMER - Kaggle Demo Smoke Test

Notebook demo này giữ cùng flow với bản full nhưng chỉ chạy rất ít batch để kiểm tra pipeline trước khi train thật. Kết quả demo không dùng để báo cáo độ chính xác.


## 1. Cấu hình demo


In [ ]:
# Paste W&B key here. Do not share this notebook publicly after pasting the key.
WANDB_API_KEY = ""
WANDB_ENTITY = None  # Optional: your W&B user/team, or keep None.

REPO_URL = "https://github.com/KhaiHASO/CNN-GNN-HMER.git"
REPO_DIR = "/kaggle/working/CNN-GNN-HMER"
PROJECT_ROOT = f"{REPO_DIR}/chuyende_tamer_temp"
BASELINE_DIR = f"{PROJECT_ROOT}/0-cnn-transformer-baseline"
CNN_GNN_DIR = f"{PROJECT_ROOT}/1-cnn-gnn"
SHARED_DATA_ROOT = "/kaggle/working/cnn_gnn_hmer_demo_data"
SHARED_CROHME_DIR = f"{SHARED_DATA_ROOT}/crohme"
OUTPUT_ROOT = "/kaggle/working/cnn_gnn_hmer_demo_outputs"

WANDB_PROJECT = "cnn-gnn-hmer"
WANDB_GROUP = "crohme-cnn-gnn-comparison-demo"
WANDB_TAGS = ["crohme", "hmer", "cnn-gnn", "kaggle", "demo-smoke-test"]

VALIDATION_YEAR = "2014"
EVAL_YEARS = ["2014"]
MAX_EPOCHS = 1
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
NUM_WORKERS = 0
MAX_SIZE = 320000
PRECISION = 16
CHECK_VAL_EVERY_N_EPOCH = 1
SAFE_CHECKPOINT_EVERY_N_EPOCHS = 1
KEEP_LOCAL_SAFE_CHECKPOINTS = 2


## 2. Đăng nhập W&B


In [ ]:
import os
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb", "pyyaml"], check=True)
import wandb

if WANDB_API_KEY.strip():
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY.strip()
else:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError("Paste WANDB_API_KEY in cell 1 or add it as Kaggle Secret.")

wandb.login(key=os.environ["WANDB_API_KEY"])


## 3. Clone repo mới nhất


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

for required in [BASELINE_DIR, CNN_GNN_DIR, f"{PROJECT_ROOT}/data/CROHME.zip"]:
    if not Path(required).exists():
        raise RuntimeError(f"Missing required path after clone: {required}")

print("Repo layout OK")
print(os.listdir(PROJECT_ROOT))


## 4. Cài Miniconda


In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -x /kaggle/working/miniconda/bin/conda ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
  bash miniconda.sh -b -f -p /kaggle/working/miniconda
  rm miniconda.sh
fi
/kaggle/working/miniconda/bin/conda --version


## 5. Tạo môi trường Python 3.7


In [ ]:
%%bash
set -e
/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true
/kaggle/working/miniconda/bin/conda create -n tamer python=3.7 -y || true
source /kaggle/working/miniconda/bin/activate tamer
python --version
pip --version


## 6. Cài dependency chung


In [ ]:
%%bash
set -e
source /kaggle/working/miniconda/bin/activate tamer
conda install pytorch-lightning=1.4.9 torchmetrics=0.6.0 pandoc=1.19.2.1 libstdcxx-ng -c conda-forge -y
pip install wandb pandas pyyaml


## 7. Giải nén dataset một lần


In [ ]:
import shutil
import subprocess
from pathlib import Path

shared_root = Path(SHARED_DATA_ROOT)
shared_crohme = Path(SHARED_CROHME_DIR)
zip_path = Path(PROJECT_ROOT) / "data" / "CROHME.zip"
extract_dir = shared_root / "extract"

if shared_crohme.exists():
    shutil.rmtree(shared_crohme)
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(extract_dir)], check=True)

candidates = [extract_dir / "CROHME_extracted" / "crohme", extract_dir / "crohme"]
source = next((p for p in candidates if p.exists()), None)
if source is None:
    dictionary = next(extract_dir.rglob("dictionary.txt"), None)
    if dictionary:
        source = dictionary.parent
if source is None:
    raise RuntimeError("Cannot locate extracted CROHME folder")

shutil.copytree(source, shared_crohme)
print("Dataset ready:", shared_crohme)
print(sorted(p.name for p in shared_crohme.iterdir()))


## 8. Tạo callback checkpoint an toàn


In [ ]:
SAFE_CALLBACK_CODE = r'''
from pathlib import Path
import pytorch_lightning as pl
try:
    import wandb
except Exception:
    wandb = None

class SafeWandbCheckpointUploader(pl.Callback):
    def __init__(self, target, run_name, every_n_epochs=1, keep_local=2):
        super().__init__()
        self.target = target
        self.run_name = run_name
        self.every_n_epochs = max(int(every_n_epochs), 1)
        self.keep_local = max(int(keep_local), 1)
        self.safe_dir = Path("safe_checkpoints") / target
        self.uploaded = set()

    def _upload(self, trainer, path, reason):
        path = Path(path)
        if not path.exists() or wandb is None or wandb.run is None:
            return
        key = f"{path.resolve()}::{path.stat().st_size}::{int(path.stat().st_mtime)}"
        if key in self.uploaded:
            return
        artifact = wandb.Artifact(
            name=f"{self.target}-{self.run_name}-safe-checkpoint",
            type="training-checkpoint",
            metadata={"target": self.target, "run_name": self.run_name, "reason": reason,
                      "epoch": int(getattr(trainer, "current_epoch", -1)),
                      "global_step": int(getattr(trainer, "global_step", -1)),
                      "source_file": path.name},
        )
        artifact.add_file(str(path), name=f"checkpoints/{path.name}")
        aliases = ["latest", self.target, reason, f"epoch-{int(getattr(trainer, 'current_epoch', -1))}"]
        wandb.run.log_artifact(artifact, aliases=aliases)
        wandb.run.log({f"safe_checkpoint/{self.target}_epoch": int(getattr(trainer, "current_epoch", -1))})
        self.uploaded.add(key)
        print(f"Uploaded safe checkpoint to W&B: {path} ({reason})")

    def _cleanup_local(self):
        ckpts = sorted(self.safe_dir.glob("*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True)
        for old in ckpts[self.keep_local:]:
            old.unlink(missing_ok=True)

    def _save_and_upload(self, trainer, reason):
        self.safe_dir.mkdir(parents=True, exist_ok=True)
        epoch = int(getattr(trainer, "current_epoch", -1))
        step = int(getattr(trainer, "global_step", -1))
        path = self.safe_dir / f"{self.target}-epoch={epoch:04d}-step={step:08d}-{reason}.ckpt"
        trainer.save_checkpoint(str(path))
        self._upload(trainer, path, reason)
        self._cleanup_local()

    def on_train_epoch_end(self, trainer, pl_module, unused=None):
        epoch_number = int(getattr(trainer, "current_epoch", 0)) + 1
        if epoch_number % self.every_n_epochs == 0:
            self._save_and_upload(trainer, "epoch")

    def on_validation_end(self, trainer, pl_module):
        for path in Path("lightning_logs").rglob("*.ckpt"):
            self._upload(trainer, path, "validation")

    def on_exception(self, trainer, pl_module, exception):
        try:
            self._save_and_upload(trainer, "exception")
        except Exception as exc:
            print(f"Failed to save exception checkpoint: {exc}")

    def on_train_end(self, trainer, pl_module):
        self._save_and_upload(trainer, "final")
        for path in Path("lightning_logs").rglob("*.ckpt"):
            self._upload(trainer, path, "final")
'''

from pathlib import Path
for project_dir in [BASELINE_DIR, CNN_GNN_DIR]:
    Path(project_dir, "safe_wandb_callback.py").write_text(SAFE_CALLBACK_CODE, encoding="utf-8")
print("Safe callback written")


## 9. Tạo config demo cho baseline và CNN-GNN


In [ ]:
import time
from pathlib import Path
import yaml

run_names = {}
config_paths = {}

def make_demo_config(project_dir, target):
    project_dir = Path(project_dir)
    cfg = yaml.safe_load(open(project_dir / "config" / "crohme.yaml", encoding="utf-8"))
    run_name = f"demo-{target}-{time.strftime('%Y%m%d-%H%M%S')}"
    cfg["trainer"]["logger"] = {
        "class_path": "pytorch_lightning.loggers.WandbLogger",
        "init_args": {
            "project": WANDB_PROJECT,
            "entity": WANDB_ENTITY,
            "name": run_name,
            "group": WANDB_GROUP,
            "job_type": f"train-{target}",
            "tags": WANDB_TAGS + [target],
            "save_dir": "lightning_logs",
            "log_model": True,
        },
    }
    cfg["trainer"]["max_epochs"] = MAX_EPOCHS
    cfg["trainer"]["gpus"] = 1
    cfg["trainer"]["precision"] = PRECISION
    cfg["trainer"]["check_val_every_n_epoch"] = CHECK_VAL_EVERY_N_EPOCH
    cfg["trainer"]["limit_train_batches"] = 1
    cfg["trainer"]["limit_val_batches"] = 1
    cfg["trainer"]["num_sanity_val_steps"] = 0

    callbacks = cfg["trainer"].setdefault("callbacks", [])
    for cb in callbacks:
        if cb.get("class_path") == "pytorch_lightning.callbacks.ModelCheckpoint":
            args = cb.setdefault("init_args", {})
            args["save_top_k"] = max(int(args.get("save_top_k", 1)), 3)
            args["save_last"] = True
            args["filename"] = f"{target}-{{epoch}}-{{step}}-{{val_ExpRate:.4f}}"
    callbacks.append({
        "class_path": "safe_wandb_callback.SafeWandbCheckpointUploader",
        "init_args": {"target": target, "run_name": run_name,
                      "every_n_epochs": SAFE_CHECKPOINT_EVERY_N_EPOCHS,
                      "keep_local": KEEP_LOCAL_SAFE_CHECKPOINTS},
    })

    cfg["data"]["folder"] = SHARED_CROHME_DIR
    cfg["data"]["test_folder"] = VALIDATION_YEAR
    cfg["data"]["max_size"] = MAX_SIZE
    cfg["data"]["train_batch_size"] = TRAIN_BATCH_SIZE
    cfg["data"]["eval_batch_size"] = EVAL_BATCH_SIZE
    cfg["data"]["num_workers"] = NUM_WORKERS

    if target == "cnn_gnn":
        cfg["model"]["use_gat"] = True
        cfg["model"].setdefault("gat_num_layers", 2)
        cfg["model"].setdefault("gat_num_heads", 8)
        cfg["model"].setdefault("gat_hidden_dim", None)
        cfg["model"].setdefault("gat_dropout", 0.1)
    else:
        for key in ["use_gat", "gat_num_layers", "gat_num_heads", "gat_hidden_dim", "gat_dropout"]:
            cfg["model"].pop(key, None)

    out = project_dir / "config" / f"kaggle_demo_{target}.yaml"
    yaml.safe_dump(cfg, open(out, "w", encoding="utf-8"), sort_keys=False)
    return str(out), run_name

config_paths["baseline"], run_names["baseline"] = make_demo_config(BASELINE_DIR, "baseline")
config_paths["cnn_gnn"], run_names["cnn_gnn"] = make_demo_config(CNN_GNN_DIR, "cnn_gnn")
print(config_paths)
print(run_names)


## 10. Cài package baseline


In [ ]:
%%bash
set -e
source /kaggle/working/miniconda/bin/activate tamer
cd /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/0-cnn-transformer-baseline
pip install -r requirements.txt
pip install -e .


## 11. Train demo baseline


In [ ]:
%%bash
set -e
source /kaggle/working/miniconda/bin/activate tamer
cd /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/0-cnn-transformer-baseline
python train.py --config config/kaggle_demo_baseline.yaml


## 12. Cài package CNN-GNN


In [ ]:
%%bash
set -e
source /kaggle/working/miniconda/bin/activate tamer
cd /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
pip install -r requirements.txt
pip install -e .


## 13. Train demo CNN-GNN


In [ ]:
%%bash
set -e
source /kaggle/working/miniconda/bin/activate tamer
cd /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
python train.py --config config/kaggle_demo_cnn_gnn.yaml


## 14. Tạo script eval demo


In [ ]:
EVAL_SCRIPT = r'''
import json
import shutil
import sys
from pathlib import Path
from pytorch_lightning import Trainer, seed_everything
from tamer.datamodule import HMEDatamodule
from tamer.lit_tamer import LitTAMER

seed_everything(7)
ckpt = Path(sys.argv[1])
data_folder = sys.argv[2]
out_dir = Path(sys.argv[3])
years = sys.argv[4:]
out_dir.mkdir(parents=True, exist_ok=True)

for year in years:
    year_dir = out_dir / year
    year_dir.mkdir(parents=True, exist_ok=True)
    trainer = Trainer(logger=False, gpus=1, limit_test_batches=1)
    dm = HMEDatamodule(folder=data_folder, test_folder=year, max_size=320000, scale_to_limit=True)
    model = LitTAMER.load_from_checkpoint(str(ckpt))
    metrics = trainer.test(model, datamodule=dm)[0]
    for name in ["result.zip", "errors.json", "predictions.json"]:
        p = Path(name)
        if p.exists():
            shutil.move(str(p), str(year_dir / name))
    (year_dir / "summary.json").write_text(json.dumps({"raw_metrics": metrics}, indent=2), encoding="utf-8")
'''

from pathlib import Path
for project_dir in [BASELINE_DIR, CNN_GNN_DIR]:
    Path(project_dir, "kaggle_demo_eval.py").write_text(EVAL_SCRIPT, encoding="utf-8")
print("Eval script written")


## 15. Eval demo baseline


In [ ]:
import subprocess
from pathlib import Path

ckpts = sorted(Path(BASELINE_DIR, "lightning_logs").rglob("*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True)
if not ckpts:
    raise RuntimeError("No baseline checkpoint found")
baseline_ckpt = ckpts[0]
print("Baseline checkpoint:", baseline_ckpt)
subprocess.run(["bash", "-lc", f"source /kaggle/working/miniconda/bin/activate tamer && cd {BASELINE_DIR} && python kaggle_demo_eval.py {baseline_ckpt} {SHARED_CROHME_DIR} {OUTPUT_ROOT}/baseline {' '.join(EVAL_YEARS)}"], check=True)


## 16. Eval demo CNN-GNN


In [ ]:
import subprocess
from pathlib import Path

ckpts = sorted(Path(CNN_GNN_DIR, "lightning_logs").rglob("*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True)
if not ckpts:
    raise RuntimeError("No CNN-GNN checkpoint found")
cnn_gnn_ckpt = ckpts[0]
print("CNN-GNN checkpoint:", cnn_gnn_ckpt)
subprocess.run(["bash", "-lc", f"source /kaggle/working/miniconda/bin/activate tamer && cd {CNN_GNN_DIR} && python kaggle_demo_eval.py {cnn_gnn_ckpt} {SHARED_CROHME_DIR} {OUTPUT_ROOT}/cnn_gnn {' '.join(EVAL_YEARS)}"], check=True)


## 17. Upload artifact eval demo lên W&B


In [ ]:
import wandb
from pathlib import Path

for target, project_dir, ckpt in [
    ("baseline", BASELINE_DIR, baseline_ckpt),
    ("cnn_gnn", CNN_GNN_DIR, cnn_gnn_ckpt),
]:
    run_name = f"{run_names[target]}-eval-upload"
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=run_name,
                     group=WANDB_GROUP, job_type=f"eval-upload-{target}",
                     tags=WANDB_TAGS + [target, "eval", "checkpoint"], reinit=True)
    artifact = wandb.Artifact(
        f"demo-{target}-{run_names[target]}-checkpoint-eval",
        type="model-eval",
        metadata={"target": target, "validation_year": VALIDATION_YEAR, "eval_years": EVAL_YEARS, "demo": True},
    )
    artifact.add_file(str(ckpt), name=f"checkpoint/{ckpt.name}")
    artifact.add_file(config_paths[target], name=f"config/{Path(config_paths[target]).name}")
    artifact.add_dir(f"{OUTPUT_ROOT}/{target}", name="eval")
    run.log_artifact(artifact)
    wandb.finish()

print("Demo completed. Outputs:", OUTPUT_ROOT)
